In [1]:
pip install booknlp

In [2]:
import torch.nn.modules.module as module_lib

_original_load_state_dict = module_lib.Module.load_state_dict

def patched_load_state_dict(self, state_dict, strict=True, **kwargs):
    return _original_load_state_dict(self, state_dict, strict=False, **kwargs)

module_lib.Module.load_state_dict = patched_load_state_dict

# Now import and run BookNLP as normal
from booknlp.booknlp import BookNLP

using device cuda


In [3]:
from booknlp.booknlp import BookNLP

model_params = {
    "pipeline": "entity,quote,supersense,event,coref",
    "model": "big"  # or "small" for faster/lighter processing
}

booknlp = BookNLP("en", model_params)

{'pipeline': 'entity,quote,supersense,event,coref', 'model': 'big'}


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

--- startup: 12.125 seconds ---


In [20]:
# Run on a plain text file
input_file = "timestamp1.txt"
output_directory = "output_folder/"
book_id = "sample_book"

booknlp.process(input_file, output_directory, book_id)

--- spacy: 3.698 seconds ---
--- entities: 20.223 seconds ---
--- quotes: 0.020 seconds ---
--- attribution: 12.266 seconds ---
--- name coref: 0.055 seconds ---
--- coref: 9.998 seconds ---
--- TOTAL (excl. startup): 46.345 seconds ---, 26698 words


In [21]:
import pandas as pd

entities = pd.read_csv("output_folder/sample_book.entities", sep="\t")
print(entities[["COREF", "start_token", "end_token", "text", "cat"]].head(20))

    COREF  start_token  end_token  \
0     105            3          4   
1      18           17         17   
2      19           35         35   
3      19           40         40   
4      20           45         45   
5     642           53         54   
6     643           63         64   
7      20           71         72   
8     644           80         80   
9      21           83         83   
10     22           85         85   
11    645           88         89   
12     23           93         93   
13     23           96         96   
14    106           99        108   
15     18          119        119   
16    644          124        124   
17    646          128        128   
18    646          134        134   
19     18          141        141   

                                                 text  cat  
0                                    Playing Pilgrims  PER  
1                                                  Jo  PER  
2                                      

In [22]:
entities = pd.read_csv("output_folder/sample_book.quotes", sep="\t")
print(entities[["quote_start", "quote_end", "mention_start", "mention_end", "mention_phrase", "char_id", "quote"]].head(20))

    quote_start  quote_end  mention_start  mention_end mention_phrase  \
0             5         15             17           17             Jo   
1            24         33             35           35            Meg   
2            44         69             71           72     little Amy   
3            79         91             93           93           Beth   
4           123        140            119          119             Jo   
5           145        149            141          141            She   
6           181        258            174          174            Meg   
7           277        344            346          346             Jo   
8           353        363            365          365           Beth   
9           384        402            404          404            Amy   
10          407        454            456          456             Jo   
11          469        494            496          496            Meg   
12          504        518            520          

In [23]:
from IPython.display import HTML, display

with open("output_folder/sample_book.book.html", "r", encoding="utf-8") as f:
    html_content = f.read()

display(HTML(html_content))

Read the necessary BookNLP outputs

In [24]:
import os
import re
import pandas as pd
from collections import Counter, defaultdict

output_directory = "output_folder"
book_id = "sample_book"

entities_file = os.path.join(
    output_directory,
    f"{book_id}.entities"
)

quotes_file = os.path.join(
    output_directory,
    f"{book_id}.quotes"
)

tokens_file = os.path.join(
    output_directory,
    f"{book_id}.tokens"
)

entities_df = pd.read_csv(entities_file, sep="\t")
quotes_df = pd.read_csv(quotes_file, sep="\t")
tokens_df = pd.read_csv(tokens_file, sep="\t")

print("Number of entity mentions:", len(entities_df))
print("Number of quotations:", len(quotes_df))
print("Number of tokens:", len(tokens_df))

Number of entity mentions: 4004
Number of quotations: 444
Number of tokens: 26698


Examine the output columns

In [25]:
print("Entity columns:")
print(entities_df.columns.tolist())

print("\nQuote columns:")
print(quotes_df.columns.tolist())

print("\nToken columns:")
print(tokens_df.columns.tolist())

Entity columns:
['COREF', 'start_token', 'end_token', 'prop', 'cat', 'text']

Quote columns:
['quote_start', 'quote_end', 'mention_start', 'mention_end', 'mention_phrase', 'char_id', 'quote']

Token columns:
['paragraph_ID', 'sentence_ID', 'token_ID_within_sentence', 'token_ID_within_document', 'word', 'lemma', 'byte_onset', 'byte_offset', 'POS_tag', 'fine_POS_tag', 'dependency_relation', 'syntactic_head_ID', 'event']


Create one canonical name for each character

In [26]:
def clean_name(name):
    name = str(name).strip()
    name = re.sub(r"\s+", " ", name)
    return name


def create_character_name_map(entities):
    data = entities.copy()

    # Retain only person entities
    if "cat" in data.columns:
        data = data[
            data["cat"].astype(str).str.upper() == "PER"
        ]

    data = data.dropna(subset=["COREF", "text"])

    data["COREF"] = data["COREF"].astype(str)
    data["text"] = data["text"].apply(clean_name)

    character_name_map = {}

    for character_id, group in data.groupby("COREF"):

        # Prefer proper names such as Elizabeth or Mr. Darcy
        proper_names = group[
            group["prop"].astype(str).str.upper() == "PROP"
        ]

        if not proper_names.empty:
            candidate_names = proper_names["text"]
        else:
            # Use nominal mentions if no proper name is available
            nominal_names = group[
                group["prop"].astype(str).str.upper() == "NOM"
            ]

            if nominal_names.empty:
                continue

            candidate_names = nominal_names["text"]

        name_counts = Counter(candidate_names)

        # Prefer frequency first and longer name second
        canonical_name = max(
            name_counts.keys(),
            key=lambda name: (
                name_counts[name],
                len(name)
            )
        )

        character_name_map[character_id] = canonical_name

    return character_name_map


character_name_map = create_character_name_map(
    entities_df
)

print("Number of characters:", len(character_name_map))

for character_id, character_name in list(
    character_name_map.items()
)[:30]:
    print(character_id, "->", character_name)

Number of characters: 522
100 -> the Hummels
1002 -> servants
1006 -> the coming guest
1008 -> half a dozen servants
1009 -> a surprised- looking servant
101 -> the Laurences
1012 -> the blancmange
1013 -> any child
1018 -> the prim old gentleman who came once to woo Aunt March
102 -> Jo
103 -> John
1031 -> the maid
1032 -> his guest
1039 -> richer friends
104 -> Italians
1043 -> his redoubtable grandfather
1047 -> the young people
1048 -> old friends
105 -> Playing Pilgrims
1052 -> such people
1055 -> her new friend
1058 -> the ` Laurence ' boy
106 -> The four young faces on which the firelight sh one
1065 -> the old man who had not forgotten him
1067 -> a young lady who knew all about the matter
107 -> each
108 -> nobody
109 -> a bookworm
110 -> no one
111 -> any one


Identify relevant token columns

In [27]:
def find_column(dataframe, possible_names):
    normalized = {
        str(column).lower(): column
        for column in dataframe.columns
    }

    for name in possible_names:
        if name.lower() in normalized:
            return normalized[name.lower()]

    raise KeyError(
        f"Could not find any of {possible_names}. "
        f"Available columns: {dataframe.columns.tolist()}"
    )


document_token_column = find_column(
    tokens_df,
    [
        "token_ID_within_document",
        "token_id_within_document",
        "token_ID"
    ]
)

sentence_id_column = find_column(
    tokens_df,
    [
        "sentence_ID",
        "sentence_id"
    ]
)

print("Document-token column:", document_token_column)
print("Sentence-ID column:", sentence_id_column)

Document-token column: token_ID_within_document
Sentence-ID column: sentence_ID


Count the sentences in every quotation

In [28]:
tokens_df[document_token_column] = pd.to_numeric(
    tokens_df[document_token_column],
    errors="coerce"
)

quotes_df["quote_start"] = pd.to_numeric(
    quotes_df["quote_start"],
    errors="coerce"
)

quotes_df["quote_end"] = pd.to_numeric(
    quotes_df["quote_end"],
    errors="coerce"
)


def count_sentences_in_quote(
    quote_start,
    quote_end,
    quote_text
):
    if pd.notna(quote_start) and pd.notna(quote_end):

        tokens_inside_quote = tokens_df[
            (
                tokens_df[document_token_column]
                >= quote_start
            )
            &
            (
                tokens_df[document_token_column]
                <= quote_end
            )
        ]

        sentence_count = (
            tokens_inside_quote[sentence_id_column]
            .dropna()
            .nunique()
        )

        if sentence_count > 0:
            return int(sentence_count)

    # Fallback if the token positions cannot be used
    quote_text = str(quote_text).strip()

    sentences = re.split(
        r"(?<=[.!?])(?:[\"'’”]*)\s+",
        quote_text
    )

    sentences = [
        sentence
        for sentence in sentences
        if sentence.strip()
    ]

    return max(1, len(sentences))

Prepare the ordered dialogue turns

In [29]:
dialogue_df = quotes_df.copy()

dialogue_df["speaker_id"] = (
    dialogue_df["char_id"]
    .astype(str)
    .str.strip()
)

dialogue_df["speaker_name"] = (
    dialogue_df["speaker_id"]
    .map(character_name_map)
)

dialogue_df["sentence_count"] = dialogue_df.apply(
    lambda row: count_sentences_in_quote(
        row["quote_start"],
        row["quote_end"],
        row["quote"]
    ),
    axis=1
)

# Remove quotes for which BookNLP did not identify
# a valid character speaker
dialogue_df = dialogue_df.dropna(
    subset=[
        "speaker_name",
        "quote_start",
        "quote_end"
    ]
)

# Put quotations in their original textual order
dialogue_df = dialogue_df.sort_values(
    by="quote_start"
).reset_index(drop=True)

display(
    dialogue_df[
        [
            "speaker_name",
            "mention_phrase",
            "quote",
            "sentence_count",
            "quote_start",
            "quote_end"
        ]
    ].head(30)
)

,speaker_name,mention_phrase,quote,sentence_count,quote_start,quote_end
0,Jo,Jo,Christmas wo n't be Christmas without any pre...,1,5,15
1,Meg,Meg,It 's so dreadful to be poor !,1,24,33
2,Amy,little Amy,I do n't think it 's fair for some girls to h...,1,44,69
3,Beth,Beth,"We 've got Father and Mother , and each other ,",1,79,91
4,Jo,Jo,"We have n't got Father , and shall not have h...",1,123,140
5,Jo,She,"perhaps never ,",1,145,149
6,Meg,Meg,You know the reason Mother proposed not havin...,3,181,258
7,Jo,Jo,But I do n't think the little we should spend...,4,277,344
8,Beth,Beth,"I planned to spend mine in new music ,",1,353,363
9,Amy,Amy,I shall get a nice box of Faber 's drawing pe...,2,384,402


Construct the weighted edges

In [30]:
MAX_TOKEN_GAP = 150

edge_weights = defaultdict(int)
exchange_evidence = []

for index in range(1, len(dialogue_df)):

    previous_quote = dialogue_df.iloc[index - 1]
    current_quote = dialogue_df.iloc[index]

    previous_speaker = previous_quote["speaker_name"]
    current_speaker = current_quote["speaker_name"]

    previous_end = previous_quote["quote_end"]
    current_start = current_quote["quote_start"]

    token_gap = current_start - previous_end

    # Same character speaking again is not an exchange
    if previous_speaker == current_speaker:
        continue

    # Ignore quotations that are too far apart
    if token_gap < 0 or token_gap > MAX_TOKEN_GAP:
        continue

    # Sort names because the network is undirected
    source, target = sorted(
        [previous_speaker, current_speaker]
    )

    # The second quotation is treated as a response
    exchanged_sentence_count = int(
        current_quote["sentence_count"]
    )

    edge_weights[(source, target)] += (
        exchanged_sentence_count
    )

    exchange_evidence.append({
        "source": source,
        "target": target,
        "previous_speaker": previous_speaker,
        "responding_speaker": current_speaker,
        "previous_quote": previous_quote["quote"],
        "response_quote": current_quote["quote"],
        "response_sentence_count": exchanged_sentence_count,
        "token_gap": int(token_gap)
    })

Produce the final edge list

In [31]:
edge_list = [
    (source, target, weight)
    for (source, target), weight
    in edge_weights.items()
]

edge_list = sorted(
    edge_list,
    key=lambda edge: edge[2],
    reverse=True
)

print("Edges in (character1, character2, weight) format:\n")

for edge in edge_list:
    print(edge)

Edges in (character1, character2, weight) format:

('Jo', 'Meg', 72)
('Beth', 'Jo', 34)
('Jo', 'Laurie', 19)
('Amy', 'Meg', 18)
('Amy', 'Jo', 17)
('Jo', 'Mrs. March', 15)
('Amy', 'Beth', 12)
('Beth', 'Meg', 10)
('Laurie', 'Mother', 9)
('Laurie', 'Meg', 7)
('Amy', 'the villain of the piece', 6)
('a tall , motherly lady', 'the most splendid mother in the world', 6)
('Jo', 'the boy', 6)
('Beth', 'Mother', 5)
('Jo', 'the young lady', 5)
('Beth', 'sir', 5)
('Meg', 'sir', 5)
('a gentleman', 'sir', 5)
('Hannah', 'Meg', 4)
('Beth', 'the girls', 4)
('Jo', 'the tall lad , whom she had imagined seventeen already', 4)
('Hannah', 'dear', 4)
('Jo', 'Mrs. Ma rch', 3)
('Hannah', 'dearies', 3)
('Jo', 'Mrs. Gardin er', 3)
('Laurie', 'the tall lad , whom she had imagined seventeen already', 3)
('Mother', 'These girls', 3)
('Jo', 'Marmee', 3)
('Meg', 'the villain of the piece', 2)
('Meg', 'the witch', 2)
('Amy', 'her mother', 2)
('Jo', 'the girls', 2)
('Mrs. March', 'the poor woman', 2)
('Don Pedro', 'Zar

Convert the edge list into a DataFrame

In [32]:
edge_list_df = pd.DataFrame(
    edge_list,
    columns=[
        "source",
        "target",
        "weight"
    ]
)

display(edge_list_df)

,source,target,weight
0,Jo,Meg,72
1,Beth,Jo,34
2,Jo,Laurie,19
3,Amy,Meg,18
4,Amy,Jo,17
...,...,...,...
72,Amy,Hannah,1
73,Meg,That boy,1
74,Meg,girls,1
75,Jo,La urie,1


Save and download the edge list

In [33]:
from google.colab import files

edge_file = "character_network_edges.csv"

edge_list_df.to_csv(
    edge_file,
    index=False
)

print("Edge list saved as:", edge_file)

files.download(edge_file)

Edge list saved as: character_network_edges.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Save evidence for manual verification

In [34]:
exchange_evidence_df = pd.DataFrame(
    exchange_evidence
)

evidence_file = "character_exchange_evidence.csv"

exchange_evidence_df.to_csv(
    evidence_file,
    index=False
)

display(exchange_evidence_df.head(20))

files.download(evidence_file)

,source,target,previous_speaker,responding_speaker,previous_quote,response_quote,response_sentence_count,token_gap
0,Jo,Meg,Jo,Meg,Christmas wo n't be Christmas without any pre...,It 's so dreadful to be poor !,1,9
1,Amy,Meg,Meg,Amy,It 's so dreadful to be poor !,I do n't think it 's fair for some girls to h...,1,11
2,Amy,Beth,Amy,Beth,I do n't think it 's fair for some girls to h...,"We 've got Father and Mother , and each other ,",1,10
3,Beth,Jo,Beth,Jo,"We 've got Father and Mother , and each other ,","We have n't got Father , and shall not have h...",1,32
4,Jo,Meg,Jo,Meg,"perhaps never ,",You know the reason Mother proposed not havin...,3,32
5,Jo,Meg,Meg,Jo,You know the reason Mother proposed not havin...,But I do n't think the little we should spend...,4,19
6,Beth,Jo,Jo,Beth,But I do n't think the little we should spend...,"I planned to spend mine in new music ,",1,9
7,Amy,Beth,Beth,Amy,"I planned to spend mine in new music ,",I shall get a nice box of Faber 's drawing pe...,2,21
8,Amy,Jo,Amy,Jo,I shall get a nice box of Faber 's drawing pe...,Mother did n't say anything about our mone y ...,3,5
9,Jo,Meg,Jo,Meg,Mother did n't say anything about our mone y ...,I know I do -- teaching those tiresome childr...,1,15


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>